In [21]:
%pip install -q pypdf ipywidgets ollama

Note: you may need to restart the kernel to use updated packages.


## Step 1: Setup & Dependencies
Install required libraries for PDF processing.

In [22]:
import sys
from pathlib import Path

sys.path.append("..")

from src.ingest import load_all_pdfs_from_folder, chunk_text

Import necessary modules and custom functions from the `src` folder.

In [23]:
documents = load_all_pdfs_from_folder("../data/raw")

print("Aantal documenten:", len(documents))
for doc in documents:
    print(doc["filename"], "->", len(doc["text"]), "characters")

Aantal documenten: 3
03_01_Explain_Schwartz.pdf -> 10842 characters
05_1_Inleiding_procedurele_SQL.pdf -> 852 characters
05_2_PLSQL.pdf -> 5257 characters


## Step 2: Load PDF Documents
Load all PDF files from the `data/raw` folder and examine their contents.

In [24]:
for doc in documents:
    print("\n" + "="*60)
    print("BESTAND:", doc["filename"])
    print("="*60)
    print(doc["text"][:1500])


BESTAND: 03_01_Explain_Schwartz.pdf
DevOps for the Database
I wrote a new ebook! 65 pages of peer
wisdom.
@ x a prb
2
Agenda
Who is this talk for?
What is EXPLAIN?
Why do I need it?
What does the output mean?
What matters most?
What tools can help me?
Where can I learn more?
@ x a prb
3
People Who Want To Use Postgres Well
It’s hard to write good queries.
EXPLAIN is a key tool to get better
at it.
But EXPLAIN is hard to use, too!
@ x a prb
5
Goal: Just Enough Knowledge
There are many detailed EXPLAIN
how-tos.
My goal today is “just enough” so
you don’t have to study multiple
in-depth sources to get started.
You should be able to use
EXPLAIN, but it’s not necessary to
be a human query planner.
I’ll give references where you can
study details as you need them.
@ x a prb
6
There’s Many Ways To Run A Query
The database has many choices:
Join order
Join type
Access type
Index choice
Algorithms
Relational databases are supposed
to  gure out how so you can say
what.
@ x a prb
8
Plans and the

Display a preview of each document's text content.

In [25]:
all_chunks = []

for doc in documents:
    chunks = chunk_text(doc["text"], chunk_size=500, overlap=100)
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "filename": doc["filename"],
            "chunk_id": i,
            "text": chunk
        })

print("Totaal aantal chunks:", len(all_chunks))
print("\nEerste chunk:\n")
print(all_chunks[0]["text"])

Totaal aantal chunks: 51

Eerste chunk:

DevOps for the Database
I wrote a new ebook! 65 pages of peer
wisdom. @ x a prb
2
Agenda
Who is this talk for? What is EXPLAIN? Why do I need it? What does the output mean? What matters most? What tools can help me? Where can I learn more? @ x a prb
3
People Who Want To Use Postgres Well
It’s hard to write good queries. EXPLAIN is a key tool to get better
at it. But EXPLAIN is hard to use, too! @ x a prb
5
Goal: Just Enough Knowledge
There are many detailed EXPLAIN
how-tos.


## Step 3: Chunk Text Documents
**Why chunking?** Large documents need to be split into smaller, manageable pieces (chunks) to:
- Create meaningful embeddings for retrieval
- Improve relevance matching for specific queries
- Handle memory and computational constraints

**Process**: Split each document into sentence-aware chunks (max 500 characters). Unlike character-based splitting, sentences are never cut mid-way, preserving context and coherence within each chunk.

In [26]:
for doc in documents:
    print(doc["filename"])
    print("Characters:", len(doc["text"]))
    print(doc["text"][:500])
    print("\n" + "-"*60 + "\n")

03_01_Explain_Schwartz.pdf
Characters: 10842
DevOps for the Database
I wrote a new ebook! 65 pages of peer
wisdom.
@ x a prb
2
Agenda
Who is this talk for?
What is EXPLAIN?
Why do I need it?
What does the output mean?
What matters most?
What tools can help me?
Where can I learn more?
@ x a prb
3
People Who Want To Use Postgres Well
It’s hard to write good queries.
EXPLAIN is a key tool to get better
at it.
But EXPLAIN is hard to use, too!
@ x a prb
5
Goal: Just Enough Knowledge
There are many detailed EXPLAIN
how-tos.
My goal today is “jus

------------------------------------------------------------

05_1_Inleiding_procedurele_SQL.pdf
Characters: 852
  
Inleiding procedurele SQL
Wim.bertels@ucll.be

  
ISO Standaard
● Sinds 1996, 1999, 2011
● SQL/PSM (Persistent Stored Modules)
● Many products adopt the standard, but none 
are 100% compatible..
– Look at the products documentation

  
Postgresql
● Everything will be a function
● Support for various other programming 
languages
– PL/pg

Summary of document statistics (character count and preview).

In [27]:
for doc in documents:
    if doc["filename"] == "05_1_Inleiding_procedurele_SQL.pdf":
        print(doc["text"])

  
Inleiding procedurele SQL
Wim.bertels@ucll.be

  
ISO Standaard
● Sinds 1996, 1999, 2011
● SQL/PSM (Persistent Stored Modules)
● Many products adopt the standard, but none 
are 100% compatible..
– Look at the products documentation

  
Postgresql
● Everything will be a function
● Support for various other programming 
languages
– PL/pgSQL (standard)
– PL/Tcl
– PL/Perl
– PL/Python
– PL/Java (uitbreiding)

  
Trusted vs Untrusted
● Trusted: restricted to what is allowed by the 
database
● Untrusted: not, take care

  
Uitbreidingen
● Debuger
● Profiler

  
● https://www.postgresql.org/docs/current/static/server-programming.html
● https://en.wikipedia.org/wiki/SQL/PSM
● “How we use Postgresql at Trustly, 'Joel Jacobson', October 2014, Madrid
Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International Public License
Referenties




Display full content of a specific document.

In [28]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

## Step 4: Create Text Embeddings
**What are embeddings?** Convert text into dense numerical vectors (384-dim for this model) that capture semantic meaning.

**Why embeddings?** Enable us to:
- Compare semantic similarity between texts
- Retrieve documents based on meaning, not just keywords
- Use cosine similarity to find the most relevant chunks

In [29]:
from sentence_transformers import SentenceTransformer

# paraphrase-multilingual-MiniLM-L12-v2 supports 50+ languages including Dutch,
# making it much better suited for Dutch course material than the English-only all-MiniLM-L6-v2
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Model geladen")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model geladen


Load the Sentence Transformer model. We use `paraphrase-multilingual-MiniLM-L12-v2` — a multilingual model supporting 50+ languages including Dutch, trained on over 1 billion sentence pairs. This replaces the English-only `all-MiniLM-L6-v2` to better handle Dutch queries and course material.

In [30]:
chunk_texts = [chunk["text"] for chunk in all_chunks]
chunk_embeddings = model.encode(chunk_texts, convert_to_numpy=True)

print("Aantal embeddings:", len(chunk_embeddings))
print("Shape van eerste embedding:", chunk_embeddings[0].shape)

Aantal embeddings: 51
Shape van eerste embedding: (384,)


Encode all text chunks into embeddings (numerical vectors).

In [31]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve_relevant_chunks(query, all_chunks, chunk_embeddings, model, top_k=3):
    query_embedding = model.encode([query], convert_to_numpy=True)
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "filename": all_chunks[idx]["filename"],
            "chunk_id": all_chunks[idx]["chunk_id"],
            "score": similarities[idx],
            "text": all_chunks[idx]["text"]
        })

    return results

## Step 5: Semantic Search Function
Define a retrieval function that:
1. Converts the user's query into an embedding
2. Calculates cosine similarity between query and all document chunks
3. Returns the top-k most similar chunks with their scores

**Score**: A similarity score between 0 and 1 (1 = perfect match, 0 = completely different).

In [33]:
query = "What is PL/SQL?"
results = retrieve_relevant_chunks(query, all_chunks, chunk_embeddings, model, top_k=5)

for i, r in enumerate(results, 1):
    print("=" * 80)
    print(f"Resultaat {i}")
    print("Bestand:", r["filename"])
    print("Chunk ID:", r["chunk_id"])
    print("Score:", round(r["score"], 4))
    print(r["text"][:700])
    print()

Resultaat 1
Bestand: 05_2_PLSQL.pdf
Chunk ID: 0
Score: 0.6102
Procedurele SQL met plpgsql
Wim.bertels@ucll.be

  
Objecten op de server
● Stored procedures
● Stored functions
● Triggers
● Def. : hoeveelheid code die opgeslagen is in de catalogus van een DB 
en die geactiveerd kan worden
● Vergelijkbaar met je in programmeren een (statische) methode zou 
noemen.

Resultaat 2
Bestand: 05_1_Inleiding_procedurele_SQL.pdf
Chunk ID: 1
Score: 0.4819
– Look at the products documentation

  
Postgresql
● Everything will be a function
● Support for various other programming 
languages
– PL/pgSQL (standard)
– PL/Tcl
– PL/Perl
– PL/Python
– PL/Java (uitbreiding)

  
Trusted vs Untrusted
● Trusted: restricted to what is allowed by the 
database
● Untrusted: not, take care

  
Uitbreidingen
● Debuger
● Profiler

  
● https://www.postgresql.org/docs/current/static/server-programming.html
● https://en.wikipedia.org/wiki/SQL/PSM
● “How we use Postgre

Resultaat 3
Bestand: 03_01_Explain_Schwartz.pdf
Chu

## Step 6: Build Context


In [34]:
def build_context(results):
    context = ""
    for i, r in enumerate(results, 1):
        context += f"[Source {i} - {r['filename']} | chunk {r['chunk_id']}]\n"
        context += r["text"] + "\n\n"
    return context

In [35]:
query = "What is PL/SQL?"
results = retrieve_relevant_chunks(query, all_chunks, chunk_embeddings, model, top_k=3)
context = build_context(results)

print(context[:2000])

[Source 1 - 05_2_PLSQL.pdf | chunk 0]
Procedurele SQL met plpgsql
Wim.bertels@ucll.be

  
Objecten op de server
● Stored procedures
● Stored functions
● Triggers
● Def. : hoeveelheid code die opgeslagen is in de catalogus van een DB 
en die geactiveerd kan worden
● Vergelijkbaar met je in programmeren een (statische) methode zou 
noemen.

[Source 2 - 05_1_Inleiding_procedurele_SQL.pdf | chunk 1]
– Look at the products documentation

  
Postgresql
● Everything will be a function
● Support for various other programming 
languages
– PL/pgSQL (standard)
– PL/Tcl
– PL/Perl
– PL/Python
– PL/Java (uitbreiding)

  
Trusted vs Untrusted
● Trusted: restricted to what is allowed by the 
database
● Untrusted: not, take care

  
Uitbreidingen
● Debuger
● Profiler

  
● https://www.postgresql.org/docs/current/static/server-programming.html
● https://en.wikipedia.org/wiki/SQL/PSM
● “How we use Postgre

[Source 3 - 03_01_Explain_Schwartz.pdf | chunk 0]
DevOps for the Database
I wrote a new ebook! 65 p

## Step 7: Build Prompt

In [43]:
def build_prompt(query, context):
    return f"""You are a study assistant that answers questions strictly from the provided course material.

Rules:
- Use ONLY the information in the context below. Do not use any outside knowledge.
- Write 2 to 3 clear, complete sentences.
- Always answer in the same language as the question.
- If the answer cannot be found in the context, respond with exactly: "This is not covered in the provided course material." or "Dit komt niet voor in het cursusmateriaal." depending on the language of the question.

Context:
{context}

Question: {query}

Answer:"""

## Step 8: Connect to Ollama
Instead of loading a model into Python directly, we use **Ollama** — a local model runner that manages the model separately. This avoids token length limits and memory management issues, and gives much better answer quality. We use `llama3.2` which handles Dutch well.

In [37]:
import ollama

# Verify Ollama is running and llama3.2 is available
models = [m.model for m in ollama.list().models]
print("Available Ollama models:", models)

if "llama3.2:latest" in models:
    print("llama3.2 is ready!")
else:
    print("llama3.2 not found — run: ollama pull llama3.2")

Available Ollama models: ['llama3.2:latest']
llama3.2 is ready!


## Step 9: Test prompt only

In [38]:
query = "What is PL/SQL?"
results = retrieve_relevant_chunks(query, all_chunks, chunk_embeddings, model, top_k=3)
context = build_context(results)
prompt = build_prompt(query, context)

print(prompt[:2500])

You are a study assistant that answers questions strictly from the provided course material.

Rules:
- Use ONLY the information in the context below. Do not use any outside knowledge.
- Write 2 to 3 clear, complete sentences.
- If the answer cannot be found in the context, respond with exactly: "This is not covered in the provided course material."

Context:
[Source 1 - 05_2_PLSQL.pdf | chunk 0]
Procedurele SQL met plpgsql
Wim.bertels@ucll.be

  
Objecten op de server
● Stored procedures
● Stored functions
● Triggers
● Def. : hoeveelheid code die opgeslagen is in de catalogus van een DB 
en die geactiveerd kan worden
● Vergelijkbaar met je in programmeren een (statische) methode zou 
noemen.

[Source 2 - 05_1_Inleiding_procedurele_SQL.pdf | chunk 1]
– Look at the products documentation

  
Postgresql
● Everything will be a function
● Support for various other programming 
languages
– PL/pgSQL (standard)
– PL/Tcl
– PL/Perl
– PL/Python
– PL/Java (uitbreiding)

  
Trusted vs Untrusted
● T

## step 10: create answer function

In [39]:
def generate_answer(prompt, model_name="llama3.2"):
    response = ollama.chat(
        model=model_name,
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"]


def answer_question(query, all_chunks, chunk_embeddings, embedding_model, top_k=3):
    results = retrieve_relevant_chunks(
        query, all_chunks, chunk_embeddings, embedding_model, top_k=top_k
    )

    context = build_context(results)
    prompt = build_prompt(query, context)
    answer = generate_answer(prompt)

    return {
        "query": query,
        "answer": answer,
        "sources": results,
        "context": context,
        "prompt": prompt
    }

## Step 11: ask first full question


In [40]:
response = answer_question(
    query="Wat is PL/SQL?",
    all_chunks=all_chunks,
    chunk_embeddings=chunk_embeddings,
    embedding_model=model,
    top_k=3
)

print("Question:")
print(response["query"])

print("\nAnswer:")
print(response["answer"])

print("\nSources:")
for s in response["sources"]:
    print(f"- {s['filename']} | chunk {s['chunk_id']} | score {s['score']:.4f}")

Question:
Wat is PL/SQL?

Answer:
This is not covered in the provided course material.

Sources:
- 05_2_PLSQL.pdf | chunk 0 | score 0.6098
- 05_1_Inleiding_procedurele_SQL.pdf | chunk 1 | score 0.4823
- 03_01_Explain_Schwartz.pdf | chunk 0 | score 0.4182


## Step 12: test multiple questions

In [46]:
test_questions = [
    "Welke programmeertalen ondersteunt PostgreSQL voor procedurele code?",
    "Wat is een stored function?",
    "Wat doet EXPLAIN in PostgreSQL?",
    "Wat is het verschil tussen een function en een procedure?",
    "Wat is het verschil tussen een walvis en een dolfijn?",
    "When do you use a function vs a procedure in PostgreSQL?",
]

for q in test_questions:
    print("=" * 80)

    response = answer_question(
        query=q,
        all_chunks=all_chunks,
        chunk_embeddings=chunk_embeddings,
        embedding_model=model,
        top_k=3
    )

    print("Question:", response["query"])
    print("Answer:", response["answer"])
    print("Top source:", response["sources"][0]["filename"])
    print()

Question: Welke programmeertalen ondersteunt PostgreSQL voor procedurele code?
Answer: PostgreSQL steunt een aantal programmeertalen voor procedurele code, waaronder PL/pgSQL (standaard), PL/Tcl, PL/Perl en PL/Python. Bovendien is er ook PL/Java beschikbaar als uitbreiding.
Top source: 05_1_Inleiding_procedurele_SQL.pdf

Question: Wat is een stored function?
Answer: Een stored function is een procedure die op de database wordt uitgevoerd en waarvan het resultaat direct teruggegeven wordt naar de aanroeper. Het heeft een specifieke return-staatus en een bepaald return-typisch. Een voorbeeld van een stored function is de "increment" function uit het boekmateriaal.
Top source: 05_2_PLSQL.pdf

Question: Wat doet EXPLAIN in PostgreSQL?
Answer: EXPLAIN in PostgreSQL helpt u om te bepalen hoe de database een query uitvoert. Het geeft een overzicht van de stap voor stap proces, waardoor u kunt beslissen of er opties zijn om de prestaties te verbeteren. EXPLAIN is een tool die u helpen kan bij 